# Lab 3 - Prepare the data, review the reward, train SFT then SFT -> RLVR

**The task:** each request gets **40 candidate products**, listed like a real catalog (name,
category, price, description - no tags). Only ~6 are real matches; the rest are traps: items just
over budget, pairs that break a multi-item total, strong interest matches that carry something the
shopper wants to avoid (or that their situation rules out), near-duplicates and filler.

**Two training stages, each with its own job:**

| Stage | Starts from | Data | Learning signal | What it adds |
|---|---|---|---|---|
| **SFT** | base model | ~700 requests **with** verified teacher answers | imitate the teacher | the task, the answer format, when to stop thinking |
| **SFT -> RLVR** | your SFT model | ~200 prompts, **no answers** | a reward function scores 8 sampled answers per prompt | better decisions on hard cases, fewer tokens |

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath("../src"))
%load_ext autoreload
%autoreload 2
import pandas as pd
pd.set_option("display.max_colwidth", 120)
from octank import config, state
S = config.get_settings("attendee")
ST = state.load(S)
print(f"region={S.region}  bucket={S.bucket}  prefix={S.prefix}  run={S.run_name}")
print(f"role={S.role_arn}")

In [ ]:
from octank import catalog, checks, formats, pools
from octank.prompts import build_prompt
from octank.pipeline import ATTENDEE, Run
run = Run(S, ATTENDEE)
cat = catalog.load_clean()
final = pd.read_parquet(run.path("final.parquet"))
sp = run.split(final)
pool_by_id = run.pools(sp, cat)
sp["split"].value_counts()

In [ ]:
ex = sp[(sp["split"] == "sft") & (sp["situation"] != "")].iloc[0].to_dict()
print("situation (never in the form):", ex["situation"], "-> avoid", list(ex["hidden_avoid"]), "\n")
print(build_prompt(ex, pool_by_id[ex["request_id"]])[:2600])

## 1. Teacher traces (rejection sampling)

In [ ]:
%%time
taught = run.teach(sp[sp["split"].isin(["sft", "holdout"])], pool_by_id)
print(f"teacher pass rate: {taught['ok'].mean():.1%} of {len(taught)}")
ok = taught[taught["ok"] == True].iloc[0]
print(ok["completion"][:2000])

`gpt-oss-120b` solves each request with its reasoning visible; code checks every answer against
every rule, including needs implied by the situation, and keeps only passing answers whose
reasoning stays under ~1,100 tokens. That guarantees the training answers are *correct* - not that
they are the *best* - and SFT can't learn beyond what the teacher solved. That's what RLVR is for.

## 2. Write the SFT, RLVR and evaluation files

In [ ]:
%%time
leftovers = set(sp.loc[sp["_flaw"].isin(["toxic", "ambiguous"]), "request_id"])
files = run.write_datasets(sp, pool_by_id, taught, exclude_ids=leftovers)
for k, v in files.items():
    print(f"{k:8s} {sum(1 for _ in open(v)):5d} rows  {v}")
print("RL prompts:", run.filter_stats["rl_prompts"])

- `sft_train.jsonl`: prompt + verified teacher answer (reasoning + JSON).
- `rlvr_train.jsonl`: prompt + the **ground truth the reward function needs**: budget, item count,
  avoid list, needs implied by the situation, every candidate's price, category, attributes and
  interests, a difficulty score and the teacher's reference answer. 20% of the RL prompts are
  freshly generated **Extreme** requests (difficulty 70+), where SFT struggles most.
- `eval.jsonl`: evaluation rows for your own holdout.

## 3. Validate with the agent skill's format detector

The `aws-ai-ml` skill ships `format_detector.py`, which recognizes and validates training and
evaluation formats. Run it on each file before training.

In [ ]:
import subprocess
FD = os.path.expanduser("~/.kiro/skills/aws-ai-ml/references/dataset-evaluation/scripts/format_detector.py")
assert os.path.exists(FD), (
    f"format_detector.py not found at {FD}. It ships with the aws-ai-ml agent skill: install the skill "
    "for the SageMaker AI agent (it lives under ~/.kiro/skills/aws-ai-ml/), then re-run this cell.")
for k in ["sft", "rlvr", "eval"]:
    r = subprocess.run([sys.executable, FD, files[k]], capture_output=True, text=True)
    lines = r.stdout.strip().splitlines()
    print(f"--- {k}: {files[k]}")
    print("\n".join(l for l in lines if l.startswith(("Format:", "Confidence:"))))
    if r.stderr.strip():
        print("stderr:", r.stderr.strip()[-1500:])
    if r.returncode != 0:
        raise RuntimeError(f"format_detector failed on {k} (exit {r.returncode})")
    valid = next((l for l in lines if l.startswith("Valid:")), None)
    print(valid or "Valid: (no 'Valid:' line in the output)")
    if not valid or "✓" not in valid:
        raise RuntimeError(f"{k} is not a valid {k} dataset:\n" + r.stdout[-2000:])

> **Agent skill:** this step follows the `aws-ai-ml` skill's **dataset-evaluation** workflow. You can ask the
> SageMaker AI agent: *"Validate my SFT and RLVR JSONL files for Qwen3.5-4B"* and it generates the equivalent cells.

In [ ]:
uris = run.upload(files)
state.update(S, datasets=uris)
uris

## 4. Review the reward function

RLVR samples 8 answers per prompt, scores each one with a **reward function**, and moves the model
toward the better answers. The reward is the most important design decision in the whole workflow:
the model will learn exactly what it pays for - including its loopholes.

Ours is a **hybrid** in `judge/reward_lambda.py`, a pyramid where higher layers only count once the
lower ones pass:

| Layer | Score |
|---|---|
| No parsable answer (cut off) | -1.0 |
| Bad ids / wrong count / duplicates | -0.6 |
| Picks an item the situation rules out (hidden need) | -0.4 |
| Picks an item the shopper said to avoid | -0.3 |
| Over budget | -0.1 |
| All rules pass | 0.30 + 0.30 x fit + 0.10 x variety + 0.30 x taste - length penalty |
| Anti-gaming | echo of the prompt -1.0, repetitive reasoning -0.4, empty "why" -0.25 |

- **Code** checks the rules exactly, and scores *fit* from the product tags the model never sees.
- **An LLM** (`gpt-oss-20b`) scores *taste* - fit, reasoning and the gift message against the
  teacher's answer - only for answers that pass every rule. If it fails, the score falls back to
  the code's fit score, never a garbage reward.
- The **length penalty** is free up to 450-900 tokens (more allowed for harder requests).

Read the function, then predict the score of each answer below before running the cell.

In [ ]:
import importlib.util

def load_reward(taste_weight="0"):
    # Load judge/reward_lambda.py. TASTE_WEIGHT=0 runs the deterministic part only (no Bedrock call).
    os.environ["TASTE_WEIGHT"] = taste_weight
    spec = importlib.util.spec_from_file_location("reward_lambda", "../judge/reward_lambda.py")
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

RW = load_reward()
print(open("../judge/reward_lambda.py").read()[:2600])

In [ ]:
# A prompt where every test answer means what its label says: the teacher solved it (a reference
# answer exists), it asks for 2+ items (so dropping one is a wrong count), and at least one
# candidate carries an attribute the shopper must avoid (explicitly or through their situation).
def usable(row):
    gt = json.loads(row["reward_model"]["ground_truth"])
    avoid = set(gt["avoid"]) | set((gt.get("implied_by_situation") or {}).get("avoid", []))
    ref = gt.get("reference")
    return (ref is not None and ref.get("picks") and int(gt["num_items"]) >= 2
            and any(avoid & set(c["attributes"]) for c in gt["candidates"].values()))

row = next((r for r in map(json.loads, open(files["rlvr"])) if usable(r)), None)
if row is None:
    raise RuntimeError(f"No RLVR prompt in {files['rlvr']} has a teacher answer, 2+ items and an avoidable "
                       "candidate - check the dataset build (section 2).")
gt = json.loads(row["reward_model"]["ground_truth"])
cands, ref = gt["candidates"], gt["reference"]
ids = [p["id"] for p in ref["picks"]]
avoid = set(gt["avoid"]) | set((gt.get("implied_by_situation") or {}).get("avoid", []))
trap = next(i for i, c in cands.items() if avoid & set(c["attributes"]) and i not in ids)
print(f"prompt: {int(gt['num_items'])} items, budget ${gt['budget_total']:.2f}, avoid {sorted(avoid)}; "
      f"trap {trap} has {sorted(avoid & set(cands[trap]['attributes']))}")

def answer(pick_ids, why="This pick suits the recipient's interests and the occasion well."):
    return json.dumps({"picks": [{"id": i, "why": why} for i in pick_ids],
                       "total": round(sum(cands[i]["price"] for i in pick_ids), 2), "gift_message": "Enjoy!"})

think = "Checking the budget, the item count, the avoid list and what the situation implies."
tests = {
    "excellent (teacher answer)": f"<think>{think}</think>\n\n{json.dumps(ref)}",
    "picks a trap item": f"<think>{think}</think>\n\n{answer(ids[:-1] + [trap])}",
    "wrong item count": f"<think>{think}</think>\n\n{answer(ids[:-1])}",       # one item short
    "empty 'why' everywhere": f"<think>{think}</think>\n\n{answer(ids, why='good')}",
    "cut off, no answer": "<think>" + think * 3,
}
batch = [{"id": k, "messages": row["prompt"] + [{"role": "assistant", "content": v}],
          "reward_model": row["reward_model"]} for k, v in tests.items()]
for r in json.loads(RW.lambda_handler({"batch": batch}, None)["body"]):
    print(f"{r['id']:28s} {r['aggregate_reward_score']:+.2f}")

**Try it:** change one rule in a copy of the function (for example drop the empty-"why" penalty or
the length penalty) and think about what a model would learn to do if you trained on it.

Locally the cell above runs the **deterministic** part only (`TASTE_WEIGHT=0`). The deployed Lambda
adds the LLM taste score, exactly as the instructor's models were trained.

## 5. Launch your training: SFT, then SFT -> RLVR

Both stages run as **background jobs on serverless model customization** - no cluster to manage.
SFT trains in ~16 minutes; SFT -> RLVR starts when SFT finishes and takes ~40 minutes, plus any
wait for serverless capacity.

> **For the rest of the session you'll use the instructor's models.** They were trained with this
> exact notebook - same code, settings and reward - on 20x more data, before the session. Lab 4
> deploys them to your endpoint right away, and Lab 5 compares them with the base model and with
> models trained at your data size. Your own jobs keep running; if your SFT model finishes in time,
> Lab 4 adds it next to the instructor's.

Evaluation of your own models is **off by default** (it would finish after the session). Set
`EVALUATE = True` below to run the same fast evaluation the instructor used.

In [ ]:
from octank import arms, reward, train, workflow

sm_session = train.session()
MPG = "octank-concierge-attendee"
model_package_group = train.ensure_model_package_group(MPG, "Octank gift concierge (attendee)")
ds = state.load(S)["datasets"]
sft_dataset_arn = train.register_dataset("octank-attendee-sft", ds["sft"])

# The reward Lambda: same code and configuration as the instructor's (Python 3.12, 300 s, 1 GB,
# gpt-oss-20b taste calls). verify_matches() proves the deployed function is the reviewed file.
# Role: config.REWARD_ROLE_ARN (env OCTANK_REWARD_ROLE_ARN) if your account provides one; otherwise
# octank creates octank-hybrid-reward-lambda-role (needs IAM create permissions).
reward_lambda = reward.deploy_hybrid_lambda(S.region, role_arn=config.REWARD_ROLE_ARN)
reward.verify_matches(reward_lambda, S.region)
state.update(S, reward_lambda=reward_lambda)
print("reward Lambda:", reward_lambda)

### 5a. Trainer recipes (hyperparameters live in `config.SFT_HP` and `config.RLVR_HP`)

In [ ]:
def make_sft(start_from):
    return workflow.sft_trainer(S, start_from, sft_dataset_arn, model_package_group, sm_session)

def make_rlvr(start_from):
    return workflow.rlvr_trainer(S, start_from, ds["rlvr"], MPG, sm_session, reward_lambda,
                                 "octank-attendee-rlvr")

print("SFT:", config.SFT_HP)
print("RLVR:", config.RLVR_HP)

### 5b. Launch

In [ ]:
EVALUATE = False                                          # True: fast-evaluate each model after training
arms.start(S, "sft", make_sft, evaluate=EVALUATE)                       # base -> SFT
arms.start(S, "sft_rlvr", make_rlvr, after="sft", evaluate=EVALUATE)    # waits for SFT, then SFT -> RLVR
arms.status(S)

In [ ]:
# Re-run any time. If the kernel restarted, re-run the cells above: arms resume from saved state.
arms.status(S)

> **Keep this kernel running until `sft_rlvr` shows `phase=training`.** The SFT -> RLVR launch waits
> for SFT in this kernel. If the kernel stops first, Lab 4 detects it and shows a one-line command
> to resume it from there.

> **Agent skill:** this step follows the `aws-ai-ml` skill's **finetuning** workflow. You can ask the
> SageMaker AI agent: *"Fine-tune Qwen3.5-4B with SFT, then continue from the SFT model with RLVR using my reward Lambda"* and it generates the equivalent cells.